# Full-map MPC with feedback from failed rollouts

Use a Colab GPU. This run keeps the previous nine goals and adds three left-room goals at (14, 12), (23, 30), and (16, 51). The original nine goals keep their start batches and planner seeds. Each goal gets up to six starts per round, with three rounds per square band. Starts may lie in either room when the band reaches it.

The adapter starts from identity. It keeps temporal pairs from successful one-macro MPC rollouts. Each failure contributes up to four comparisons with its original start, using distinct later frames spread across the rollout. The start must have lower cost than each sampled frame. There is no per-step distance filter or success quota. It carries forward on an empty round. The default collects through Band 4 and schedules 864 training rollouts. This uses the same twelve goals, starts, and planner seeds as the success-only control. Set `RESUME_RUN_ID` to continue this feedback run.

The fixed benchmark replays the previous full-map run's Band 4 Round 1 cases using this run's frozen adapter after Band 3. It reuses the starts, goals, planner seeds, and cached goal latents. These 36 evaluation rollouts contribute no training pairs. The tables also compare against the saved twelve-goal success-only run when available.

The overlays show the cost used to collect each round, using the actual collection goal. The held-out evaluation also measures the cost geometry on the same eight reference goals as before.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
RUNS=DRIVE/'datasets/curriculum_cost_adapter'
RIGHT_GOALS_RUN=RUNS/'experience_mpc_repeated_half_square_v1/20261108T162811962955Z'
BENCHMARK_RUN=RUNS/'experience_mpc_repeated_full_square_v1/20261108T173436313734Z'
RUN_BENCHMARK=True
FAILURE_PAIRS=4
CONTROL_RUN=RUNS/'experience_mpc_twelve_goals_full_square_v1/20261108T193815524623Z'
EXTRA_LEFT_GOALS=[(14,12),(23,30),(16,51)]
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
ROUNDS_PER_BAND=3
LAST_BAND=4
PLOT_GOALS=[4,9,10,11]   # One right-room goal and the three added left-room goals
PLOT_BANDS=[3,4]       # Applies to trajectory plots and heatmaps
HEATMAP_GOALS=[0,4]      # Held-out evaluation goals, one per room
DRAW_COST_MAPS=True
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_mpc_failure_feedback_full_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-full-failure-feedback')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='387b158a1b71b51fb320b22aa773ffb7c1c603ba'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the frozen encoder, world model and renderer

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_square',operation,
          '--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL,'--rounds-per-band',ROUNDS_PER_BAND,
          '--right-goals-run',RIGHT_GOALS_RUN,'--failure-pairs',FAILURE_PAIRS]
    for goal in EXTRA_LEFT_GOALS:
        args+=['--extra-left-goal',*goal]
    if operation=='benchmark':
        args+=['--benchmark-run',BENCHMARK_RUN]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Collect and fit

Collection finishes all three rounds of Band 4 and ends automatically. Each completed round is saved separately. Use the same run ID to reuse completed work.

In [ ]:
assert ROUNDS_PER_BAND>=2 and 1<=LAST_BAND<=4
for band in range(1,LAST_BAND+1):
    run_band(band)

## Check collection by round and route

A crossing counts when the ball passes through the doorway and reaches the other room. A rollout can cross and still fail to reach the goal. Square entry measures whether the ball entered the previous square by action 5 or at any later point. Crossing rates apply to opposite-room starts. Doorway pairs are retained consecutive simulator transitions along the opening. Failure pairs are reported separately. They can span several simulator steps.

In [ ]:
from IPython.display import Image, display
import numpy as np
import pandas as pd

def read_json(path):
    return json.loads(path.read_text())

manifest=read_json(OUTPUT/'experiment.json')['manifest']
stages={}
for band in range(1,LAST_BAND+1):
    for number in range(1,ROUNDS_PER_BAND+1):
        path=OUTPUT/f'band_{band}_round_{number}'/'complete.json'
        if path.is_file():
            stages[band,number]=read_json(path)
if not stages:
    raise RuntimeError('Finish at least one collection round first.')
print('Collection goals:',[tuple(round(v,2) for v in g) for g in manifest['goals']])

rows=[]
for (band,number),stage in stages.items():
    for row in stage['collection']:
        route=row['case']['route']
        cross_room=route in ('left_to_right','right_to_left')
        diag=row['boundary']
        rows.append(dict(band=band,round=number,goal=row['case']['goal_index'],route=route,
            success=row['success'],crossed=row['reached_other_room'] if cross_room else np.nan,
            entered_by_5=diag['entered_by_five_actions'] if band>1 else np.nan,
            entered=diag['first_entry_step'] is not None if band>1 else np.nan,
            pairs=row['retained_pairs'],failure_pairs=row['retained_pairs'] if not row['success'] else 0,
            doorway_pairs=row['retained_doorway_pairs']))
data=pd.DataFrame(rows)

def summarize(grouped):
    return grouped.agg(attempts=('success','size'),successes=('success','sum'),
        success_rate=('success','mean'),crossed=('crossed',lambda x:x.sum(min_count=1)),
        crossing_rate=('crossed','mean'),entered_by_5=('entered_by_5',lambda x:x.sum(min_count=1)),
        entered_ever=('entered',lambda x:x.sum(min_count=1)),
        new_pairs=('pairs','sum'),failure_pairs=('failure_pairs','sum'),
        doorway_pairs=('doorway_pairs','sum')).round(3)

rounds=summarize(data.groupby(['band','round']))
rounds['cumulative_pairs']=rounds.new_pairs.cumsum()
rounds['cumulative_failure_pairs']=rounds.failure_pairs.cumsum()
rounds['updates']=[stages[key]['updates'] for key in rounds.index]
display(rounds)
by_route=summarize(data.groupby(['band','round','route']))
print('Opposite-room starts by direction')
display(by_route[by_route.index.get_level_values('route').isin(['left_to_right','right_to_left'])])
rounds.to_csv(OUTPUT/'round_summary.csv')
by_route.to_csv(OUTPUT/'round_summary_by_route.csv')

paired=[]
for (band,number),stage in stages.items():
    path=CONTROL_RUN/f'band_{band}_round_{number}'/'complete.json'
    if not path.is_file():
        continue
    reference={r['case']['case_id']:r for r in read_json(path)['collection']}
    for row in stage['collection']:
        before=reference.get(row['case']['case_id'])
        if before is None or before['case']!=row['case']:
            continue
        paired.append(dict(band=band,round=number,route=row['case']['route'],cases=1,
            success_only=int(before['success']),with_feedback=int(row['success']),
            wins=int(row['success'] and not before['success']),
            losses=int(before['success'] and not row['success'])))
if paired:
    pairs=pd.DataFrame(paired)
    print('Matched starts against the twelve-goal success-only run')
    display(pairs.groupby(['band','round']).sum(numeric_only=True))
    pairs.groupby(['band','round','route']).sum(numeric_only=True).to_csv(OUTPUT/'paired_collection_by_route.csv')

## Replay the same Band 4 starts

The benchmark compares the incoming Band 4 adapters on the exact same cases. It shows the original six-goal baseline and the twelve-goal success-only baseline when available. The right-room reference goals in this benchmark come from the earlier full-map run.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

def wall_overlay(ax):
    for y,height in ((4,22),(34,26)):
        ax.add_patch(Rectangle((29,y),6,height,facecolor='#555',alpha=.75,zorder=2))

def square_edges(ax, goal, band):
    gx,gy=goal
    for radius,style in zip(band,('-','--')):
        if radius:
            ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,fill=False,
                                   edgecolor='#db9123',linestyle=style,linewidth=1.2,zorder=3))


if RUN_BENCHMARK and (3,ROUNDS_PER_BAND) in stages:
    shell(command('benchmark'),cwd=REPO)
    benchmark=read_json(OUTPUT/'band_4_fixed_benchmark.json')
    control_path=CONTROL_RUN/'band_4_fixed_benchmark.json'
    control={r['case']['case_id']:r for r in read_json(control_path)['rows']} if control_path.is_file() else {}
    rows=[]
    for row in benchmark['rows']:
        case=row['case']
        route=case['route']
        if route=='same_room':
            route='same_room_left' if case['goal'][0]<29 else 'same_room_right'
        before=control.get(case['case_id'])
        matched=before is not None and before['case']==case
        rows.append(dict(route=route,cases=1,six_goal_success=int(row['previous_success']),
                         success_only=int(before['success']) if matched else None,
                         with_feedback=int(row['success']),
                         six_goal_crossed=int(row['previous_crossed']),
                         success_only_crossed=int(before['crossed']) if matched else None,
                         feedback_crossed=int(row['crossed'])))
    table=pd.DataFrame(rows).groupby('route').sum(min_count=1)
    table.loc['all']=table.sum(min_count=1)
    display(table)
    table.to_csv(OUTPUT/'band_4_fixed_benchmark_summary.csv')
elif RUN_BENCHMARK:
    print('Complete Band 3 to run the fixed Band 4 benchmark.')

## Inspect the cost geometry on held-out goals

The table compares ordering against shortest legal path distance. It separates the whole map from positions in the room opposite the goal. The maps use median-normalized latent MSE on a common scale for each goal. Each row starts with the incoming cost and continues through that band's rounds. The panels mark zero-update rounds as carried forward.

In [ ]:
if DRAW_COST_MAPS:
    finished=[b for b in range(1,LAST_BAND+1) if (b,ROUNDS_PER_BAND) in stages]
    if not finished:
        raise RuntimeError('Finish all rounds of at least one band to compare costs.')
    last=max(finished)
    comparison=OUTPUT/'comparison.json'
    required={(b,r) for b in range(1,last+1) for r in range(1,ROUNDS_PER_BAND+1)}
    report=read_json(comparison) if comparison.is_file() else {'stages':[]}
    saved={(s['band'],s['round']) for s in report['stages']}
    if 'collection_raw_grid' not in report or not required.issubset(saved):
        shell(command('compare',last),cwd=REPO)
        report=read_json(comparison)
    report['stages']=[s for s in report['stages'] if (s['band'],s['round']) in required]
    rows=[]
    for stage in report['stages']:
        for i,goal in enumerate(stage['by_goal']):
            for cohort in ('full_map','opposite_room','local','local_opposite_room'):
                for method,result in goal['metrics'][cohort].items():
                    rows.append(dict(band=stage['band'],round=stage['round'],goal=i,
                        cohort=cohort,method=method,**result))
    metrics=pd.DataFrame(rows)
    metrics[['correct','uphill']]*=100
    display(metrics.pivot_table(index=['band','round'],columns=['cohort','method'],values='correct').round(2))
    metrics.to_csv(OUTPUT/'heldout_ranking_by_round.csv',index=False)

    axis=report['axis']
    shape=(len(axis),len(axis))
    plotted=[b for b in PLOT_BANDS if b in finished]
    plots=OUTPUT/'heatmaps'
    plots.mkdir(exist_ok=True)
    checkpoints={(s['band'],s['round']):s for s in report['stages']}
    for index in HEATMAP_GOALS if plotted else []:
        goal=report['goals'][index]
        maps={(0,0):np.asarray(report['raw_grid'][index],dtype=float).reshape(shape)}
        maps.update({key:np.asarray(s['by_goal'][index]['grid_costs'],dtype=float).reshape(shape)
                     for key,s in checkpoints.items()})
        medians={key:float(np.nanmedian(cost)) for key,cost in maps.items()}
        scaled={key:cost/max(medians[key],1e-9) for key,cost in maps.items()}
        vmax=max(float(np.nanmax(cost)) for cost in scaled.values())
        cols=ROUNDS_PER_BAND+1
        fig,axes=plt.subplots(len(plotted),cols,figsize=(3.4*cols,3.8*len(plotted)),
                              squeeze=False,layout='constrained')
        for row,band in enumerate(plotted):
            incoming=(0,0) if band==1 else (band-1,ROUNDS_PER_BAND)
            keys=[incoming,*[(band,n) for n in range(1,cols)]]
            lo,hi=manifest['bands'][band-1]
            for number,(ax,key) in enumerate(zip(axes[row],keys)):
                mesh=ax.pcolormesh(axis,axis,scaled[key],shading='nearest',vmin=0,vmax=vmax,cmap='viridis')
                wall_overlay(ax)
                square_edges(ax,goal,(lo,hi))
                ax.scatter(*goal,marker='*',s=82,c='black',edgecolor='white',zorder=5)
                title='Incoming raw MSE' if key==(0,0) else 'Incoming' if not number else f'After round {number}'
                if number and not checkpoints[key]['updates']:
                    title+=' (carried forward)'
                ax.set(title=f'{title}\nMedian MSE {medians[key]:.3g}',xlim=(4,60),ylim=(4,60),
                       aspect='equal',xlabel='Ball x')
                if not number:
                    ax.set_ylabel(f'Band {band} ({lo}–{hi})\nBall y')
        fig.colorbar(mesh,ax=axes.ravel().tolist(),shrink=.6,label='MSE divided by map median')
        fig.suptitle(f'Held-out goal {index} {tuple(goal)}')
        path=plots/f'goal_{index}_rounds.png'
        fig.savefig(path,dpi=120,bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))

## Overlay the rollouts on the incoming planner cost

Each panel shows the cost of rendered positions to the **actual collection goal**, using the frozen adapter that collected that round. Colors show latent MSE, with a separate absolute colorbar per panel. The star is the goal and amber outlines are the square-band edges.

Green shows successes, with white dots at retained training observations. Magenta dots mark later failure frames ranked worse than the start. The start of each sampled failure is circled. Red shows the first five actions of failures, and dashed red joins their start and end. An amber diamond marks the end of the first macro for each rollout, or the final observation if it succeeded early.

In [ ]:
if DRAW_COST_MAPS:
    plot_dir=OUTPUT/'collection_overlays'
    plot_dir.mkdir(exist_ok=True)
    for band in PLOT_BANDS:
        if band not in finished:
            continue
        for goal_index in PLOT_GOALS:
            groups=[]
            for number in range(1,ROUNDS_PER_BAND+1):
                batch=[r for r in stages[band,number]['collection'] if r['case']['goal_index']==goal_index]
                if batch:
                    groups.append((number,batch))
            if not groups:
                continue
            goal=manifest['goals'][goal_index]
            fig,axes=plt.subplots(1,len(groups),figsize=(5.25*len(groups),5.1),
                                  squeeze=False,layout='constrained')
            for ax,(number,episodes) in zip(axes[0],groups):
                incoming=(band,number-1) if number>1 else (band-1,ROUNDS_PER_BAND)
                if incoming[0]==0:
                    values=report['collection_raw_grid'][goal_index]
                    source='Raw MSE'
                else:
                    values=checkpoints[incoming]['collection_grid'][goal_index]
                    source=f'After Band {incoming[0]} Round {incoming[1]}'
                field=np.asarray(values,dtype=float).reshape(shape)
                mesh=ax.pcolormesh(axis,axis,field,shading='nearest',cmap='viridis',
                                   vmin=0,vmax=max(float(np.nanmax(field)),1e-9))
                wall_overlay(ax)
                square_edges(ax,goal,manifest['bands'][band-1])
                for row in sorted(episodes,key=lambda r:r['success']):
                    path=np.asarray(row['boundary']['positions'])
                    color='#64f39a' if row['success'] else '#ff727a'
                    shown=path if row['success'] else path[:6]
                    ax.plot(*shown.T,color=color,alpha=.88,linewidth=1.6,zorder=4)
                    if not row['success']:
                        ax.plot(*path[[0,-1]].T,color=color,linestyle='--',alpha=.6,zorder=4)
                    retained=np.asarray(row['retained_positions']).reshape(-1,2)
                    if len(retained):
                        points=retained if row['success'] else retained[1:]
                        ax.scatter(*points.T,s=17,facecolors='white' if row['success'] else '#f489fc',
                                   edgecolors='#252526',linewidths=.55,zorder=5)
                    ax.scatter(*path[min(5,len(path)-1)],s=28,marker='D',color='#ffc341',
                               edgecolor='#252526',linewidths=.5,zorder=6)
                    ax.scatter(*path[0],s=42,facecolors='none',edgecolors=color,zorder=7)
                    ax.scatter(*path[-1],s=42,color=color,marker='x',zorder=7)
                ax.scatter(*goal,marker='*',s=125,c='black',edgecolors='white',zorder=8)
                ok=sum(r['success'] for r in episodes)
                ax.set(title=f'Round {number} · {ok}/{len(episodes)} success\n{source}',
                       xlim=(4,60),ylim=(4,60),aspect='equal',xlabel='Ball x',ylabel='Ball y')
                fig.colorbar(mesh,ax=ax,shrink=.68,label='Latent MSE')
            fig.suptitle(f'Collection goal {goal_index} ({goal[0]:.2f}, {goal[1]:.2f}) · Band {band}')
            fig.legend(handles=[
                Line2D([],[],color='#1b984e',label='Success'),
                Line2D([],[],color='#c44048',label='Failed first five'),
                Line2D([],[],color='#c44048',ls='--',label='Failed start to end'),
                Line2D([],[],marker='o',mfc='white',mec='#252526',ls='none',label='Training observation'),
                Line2D([],[],marker='o',mfc='#f489fc',mec='#252526',ls='none',label='Sampled failure frame'),
                Line2D([],[],marker='D',mfc='#ffc341',mec='#252526',ls='none',label='First macro endpoint'),
            ],loc='upper center',bbox_to_anchor=(.5,-.005),ncol=3,frameon=False)
            path=plot_dir/f'goal_{goal_index}_band_{band}.png'
            fig.savefig(path,dpi=140,bbox_inches='tight')
            plt.close(fig)
            display(Image(filename=str(path)))